In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telecom_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)


In [ ]:
# Q6: Churn rate by Gender, Senior Citizen, Partner, Dependents
churn_dimensions = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']

results = []
for dim in churn_dimensions:
    group = df.groupby(dim)['Churn'].apply(
        lambda x: (x == 'Yes').sum() / len(x) * 100
    ).reset_index()
    group.columns = ['value', 'churn_rate']
    group['dimension'] = dim
    results.append(group)

report = pd.concat(results).sort_values('churn_rate', ascending=False)
report['churn_rate'] = report['churn_rate'].round(2)
report[['dimension', 'value', 'churn_rate']]

In [ ]:
# Q7: Customer segments based on tenure and churn analysis
df['tenure_segment'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['New (0-12)', 'Growing (12-24)', 'Mature (24-48)', 'Loyal (48-72)']
)

segment_analysis = df.groupby('tenure_segment').agg(
    customer_count = ('customerID', 'count'),
    churned        = ('Churn', lambda x: (x == 'Yes').sum()),
    churn_rate     = ('Churn', lambda x: round((x == 'Yes').sum() / len(x) * 100, 2))
).reset_index()

segment_analysis

In [ ]:
# Q8: Top 10 customer profiles most likely to churn
profiles = df.groupby(['Contract', 'PaymentMethod', 'InternetService']).agg(
    customer_count = ('customerID', 'count'),
    churned        = ('Churn', lambda x: (x == 'Yes').sum()),
    churn_rate     = ('Churn', lambda x: round((x == 'Yes').sum() / len(x) * 100, 2))
).reset_index()

profiles = profiles[profiles['customer_count'] >= 30]

profiles.sort_values('churn_rate', ascending=False).head(10)

In [ ]:
# Q9: Calculate CLV and compare churned vs retained customers
df['CLV'] = df['MonthlyCharges'] * df['tenure']

clv_comparison = df.groupby('Churn').agg(
    customer_count = ('customerID', 'count'),
    avg_clv        = ('CLV', lambda x: round(x.mean(), 2)),
    avg_tenure     = ('tenure', lambda x: round(x.mean(), 2)),
    avg_monthly    = ('MonthlyCharges', lambda x: round(x.mean(), 2))
).reset_index()

clv_comparison

In [ ]:
# Q10: Customer Risk Score using Contract, Tenure, Tech Support, Online Security
def calculate_risk_score(row):
    score = 0

    if row['Contract'] == 'Month-to-month':
        score += 3
    elif row['Contract'] == 'One year':
        score += 1
    else:
        score += 0

    if row['tenure'] <= 12:
        score += 3
    elif row['tenure'] <= 24:
        score += 2
    elif row['tenure'] <= 48:
        score += 1
    else:
        score += 0

    if row['TechSupport'] == 'No':
        score += 2

    if row['OnlineSecurity'] == 'No':
        score += 2

    return score

df['risk_score'] = df.apply(calculate_risk_score, axis=1)

df.groupby('risk_score')['Churn'].apply(
    lambda x: round((x == 'Yes').sum() / len(x) * 100, 2)
).reset_index().rename(columns={'Churn': 'churn_rate'})